# L08 · Actor-Critic, GAE, and PPO from Scratch

## Goal

- compute critic targets and GAE
- distinguish old and current ratios
- verify PPO clipping for both advantage signs

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L08:toy:42").hexdigest()
print(f"lesson=L08 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L08 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:f21ea10a7ab463f89b492a9a914c94806bcb3fb04b187ae947a27554b31f76be data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: REINFORCE → **Actor-Critic, GAE, and PPO** → LLM policy

$$\hat A_t=\delta_t+(\gamma\lambda)\hat A_{t+1},\qquad L^{clip}=\min(r_t\hat A_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)\hat A_t)$$

The critic predicts state value, and TD residual `δ` supplies advantage estimates. GAE lambda interpolates between short bootstraps and long returns. The PPO ratio compares current selected-action probability with the rollout-time old policy.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** At ratio 1.25 and epsilon 0.2, are positive and negative advantages clipped in the same way? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. The `min` caps positive advantage at 1.2, while negative advantage keeps the worse -1.25. Audit both signs separately.</details>

In [2]:
from rl_study.algorithms.ppo import ppo_policy_loss
from rl_study.math import generalized_advantage_estimate
rewards = torch.tensor([0.0, 1.0])
values = torch.tensor([0.2, 0.4, 0.0])
terminated = torch.tensor([False, True])
truncated = torch.tensor([False, False])
gae, gae_returns = generalized_advantage_estimate(
    rewards, values, terminated, truncated, gamma=0.9, gae_lambda=0.95
)
old_logp = torch.zeros(2)
current_logp = torch.log(torch.tensor([1.25, 1.25]))
ppo_output = ppo_policy_loss(current_logp, old_logp, torch.tensor([1.0, -1.0]))
print({"gae": gae.tolist(), "returns": gae_returns.tolist(),
       "ratios": ppo_output.ratio.tolist(),
       "clipped": ppo_output.clipped_objective.tolist()})

{'gae': [0.6729999780654907, 0.6000000238418579], 'returns': [0.8729999661445618, 1.0], 'ratios': [1.25, 1.25], 'clipped': [1.2000000476837158, -1.2000000476837158]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Old log-probability is a detached rollout snapshot. KL penalties and early stopping are alternatives; clipping alone should not be interpreted as a guaranteed trust region.

**Common trap:** Recomputing old log-probability with the current policy makes the ratio always 1 and destroys update diagnostics. Preserve rollout log-probabilities and policy version. Regression tests: `test_gae_analytic`, `test_ppo_clip_sign_cases`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert torch.allclose(ppo_output.ratio, torch.tensor([1.25, 1.25]))
assert torch.isfinite(gae).all()
print("checks=passed")

checks=passed


**Recall:** What targets do lambda=0 and lambda=1 approach? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** GAE produced two finite advantages and both ratios were 1.25. The printed clipped terms are asymmetric across advantage signs.
- Executable checks: `test_gae_analytic`, `test_ppo_clip_sign_cases`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L09 replaces one PPO action with LLM response tokens and redefines KL, masks, and reward placement.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/classic.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `gae-2015` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `repo-spinningup` — `docs/sources.yml`